# Notebook 01: Data Preparation and Cleaning

**Project:** Retail Sales Intelligence  
**Industry:** Retail and Consumer Electronics  
**Tools:** Python, Pandas, NumPy, Matplotlib  
**Stage:** Data Preparation and Data Quality Assessment

## 1.1 Introduction

This notebook focuses on preparing and validating the datasets used in the Retail Sales Intelligence project.

The project uses the Global Electronics Retailer dataset to investigate retail sales performance, product profitability, customer purchasing behaviour and geographical sales trends.

Before conducting exploratory analysis and developing the Power BI dashboard, the source datasets must be inspected, cleaned and validated to ensure data quality and consistency.

## 1.2 Objectives

The objectives of this notebook are to:

1. Load and inspect the source datasets.
2. Understand the structure and characteristics of each dataset.
3. Identify missing values, duplicate records and inconsistencies.
4. Standardise column names, text values and data types.
5. Apply appropriate data-cleaning procedures.
6. Validate the cleaned datasets.
7. Export the prepared datasets for subsequent analysis.

## 1.3 Datasets

The project uses five primary datasets:

- Sales: Transaction-level sales records.
- Customers: Customer demographic and geographical information.
- Products: Product descriptions, categories, prices and costs.
- Stores: Store information and geographical details.
- Exchange Rates: Currency exchange rates over time.

A separate Data Dictionary is used as reference documentation.

## 1.4 Methodology

The data preparation process follows a structured workflow:

**Data Loading → Initial Inspection → Data Quality Assessment → Data Cleaning → Validation → Export**

## 2. Importing Libraries

The following Python libraries are used throughout the notebook:

- **Pandas:** Data manipulation, inspection and cleaning.
- **NumPy:** Numerical operations and missing-value handling.
- **Matplotlib:** Basic data visualisation.
- **Pathlib:** File and directory management.
- **Regular Expressions:** Standardisation of column names.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

from pathlib import Path
from IPython.display import display

# Display configuration
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries imported successfully.")

Libraries imported successfully.


## 3. Project Directory Configuration

The project uses a structured directory system to separate raw datasets from processed datasets.

The original source files will remain unchanged in the `data/raw/` directory, while cleaned datasets will be exported to `data/processed/`.

Relative file paths are used to improve project portability.

In [3]:
# Define project directories
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Create processed directory if it does not exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Raw data directory:", RAW_DIR.resolve())
print("Processed data directory:", PROCESSED_DIR.resolve())

# Confirm that the raw directory exists
if not RAW_DIR.exists():
    raise FileNotFoundError(
        "Raw data directory not found. Check your project folder structure."
    )

Project root: C:\Users\Administrator\retail-sales-intelligence
Raw data directory: C:\Users\Administrator\retail-sales-intelligence\data\raw
Processed data directory: C:\Users\Administrator\retail-sales-intelligence\data\processed


## 4. Data Loading

The five primary datasets are imported from the raw data directory using Pandas.

The Data Dictionary is also loaded separately to provide reference information about the available fields.

Each dataset is stored in a dictionary to simplify subsequent inspection and cleaning operations.

In [5]:
# Define source filenames
file_paths = {
    "sales": "Sales.csv",
    "customers": "Customers.csv",
    "products": "Products.csv",
    "stores": "Stores.csv",
    "exchange_rates": "Exchange_Rates.csv"
}

# Function to load CSV files
def load_csv(file_path):
    try:
        return pd.read_csv(file_path, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(
            file_path,
            encoding="latin-1",
            low_memory=False
        )

# Verify that all expected files exist
for filename in file_paths.values():
    if not (RAW_DIR / filename).exists():
        raise FileNotFoundError(f"Missing file: {filename}")

# Load datasets
raw_data = {
    name: load_csv(RAW_DIR / filename)
    for name, filename in file_paths.items()
}

# Load data dictionary
dictionary_path = RAW_DIR / "Data_Dictionary.csv"

data_dictionary = (
    load_csv(dictionary_path)
    if dictionary_path.exists()
    else None
)

print("DATASETS LOADED SUCCESSFULLY")

for name, df in raw_data.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

if data_dictionary is not None:
    print("\nData Dictionary:")
    display(data_dictionary.head())

DATASETS LOADED SUCCESSFULLY
sales: 62,884 rows, 9 columns
customers: 15,266 rows, 10 columns
products: 2,517 rows, 10 columns
stores: 67 rows, 5 columns
exchange_rates: 11,215 rows, 3 columns

Data Dictionary:


,Table,Field,Description
0,Sales,Order Number,Unique ID for each order
1,Sales,Line Item,Identifies individual products purchased as pa...
2,Sales,Order Date,Date the order was placed
3,Sales,Delivery Date,Date the order was delivered
4,Sales,CustomerKey,Unique key identifying which customer placed t...


## 5. Initial Data Inspection

An initial inspection is conducted to understand the structure of each dataset.

The inspection includes:

- Dataset dimensions.
- Column names.
- Data types.
- Sample records.
- Descriptive statistics.

This provides an overview of the available information and helps identify potential data quality issues before cleaning begins.

In [7]:
for name, df in raw_data.items():

    print("\n" + "=" * 70)
    print(f"{name.upper()} DATASET")
    print("=" * 70)

    print("\nDataset dimensions:")
    print(df.shape)

    print("\nColumn names:")
    print(df.columns.tolist())

    print("\nFirst five records:")
    display(df.head())

    print("\nData types:")
    display(df.dtypes.to_frame("Data Type"))

    print("\nDescriptive statistics:")
    display(df.describe(include="all").T)


SALES DATASET

Dataset dimensions:
(62884, 9)

Column names:
['Order Number', 'Line Item', 'Order Date', 'Delivery Date', 'CustomerKey', 'StoreKey', 'ProductKey', 'Quantity', 'Currency Code']

First five records:


,Order Number,Line Item,Order Date,Delivery Date,CustomerKey,StoreKey,ProductKey,Quantity,Currency Code
0,366000,1,1/1/2016,NaN,265598,10,1304,1,CAD
1,366001,1,1/1/2016,1/13/2016,1269051,0,1048,2,USD
2,366001,2,1/1/2016,1/13/2016,1269051,0,2007,1,USD
3,366002,1,1/1/2016,1/12/2016,266019,0,1106,7,CAD
4,366002,2,1/1/2016,1/12/2016,266019,0,373,1,CAD



Data types:


,Data Type
Order Number,int64
Line Item,int64
Order Date,object
Delivery Date,object
CustomerKey,int64
StoreKey,int64
ProductKey,int64
Quantity,int64
Currency Code,object



Descriptive statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Order Number,"62,884.00",NaN,NaN,NaN,"1,430,904.91","453,296.30","366,000.00","1,121,017.00","1,498,015.50","1,788,010.00","2,243,032.00"
Line Item,"62,884.00",NaN,NaN,NaN,2.16,1.37,1.00,1.00,2.00,3.00,7.00
Order Date,62884,1641,12/21/2019,222,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Delivery Date,13165,1492,1/6/2020,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CustomerKey,"62,884.00",NaN,NaN,NaN,"1,180,797.46","585,963.40",301.00,"680,858.00","1,261,200.50","1,686,496.00","2,099,937.00"
StoreKey,"62,884.00",NaN,NaN,NaN,31.80,22.98,0.00,8.00,37.00,53.00,66.00
ProductKey,"62,884.00",NaN,NaN,NaN,"1,125.86",709.24,1.00,437.00,"1,358.00","1,650.00","2,517.00"
Quantity,"62,884.00",NaN,NaN,NaN,3.14,2.26,1.00,1.00,2.00,4.00,10.00
Currency Code,62884,5,USD,33767,NaN,NaN,NaN,NaN,NaN,NaN,NaN


... (20 outputs left out)


Data types:


,Data Type
Date,object
Currency,object
Exchange,float64



Descriptive statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Date,11215,2243,1/1/2015,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Currency,11215,5,USD,2243,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Exchange,"11,215.00",NaN,NaN,NaN,1.06,0.25,0.63,0.86,1.00,1.31,1.73


## 6. Data Quality Assessment

This section evaluates the quality and completeness of the source datasets.

The following checks are performed:

1. Missing-value identification.
2. Missing-value percentages.
3. Exact duplicate detection.
4. Potential identifier inspection.
5. Identification of empty or unnecessary columns.

These checks establish a baseline against which the effects of subsequent cleaning procedures can be measured.

Repeated identifiers are not automatically treated as duplicates because multiple records may legitimately relate to the same customer, product or store.

In [8]:
# Function to generate missing-value reports
def missing_value_report(df):

    report = pd.DataFrame({
        "Missing Count": df.isna().sum(),
        "Missing Percentage": (
            df.isna().mean() * 100
        ).round(2),
        "Data Type": df.dtypes.astype(str)
    })

    return report.sort_values(
        by="Missing Percentage",
        ascending=False
    )


# Generate quality reports
quality_summary = []

for name, df in raw_data.items():

    print(f"\n{name.upper()} — MISSING VALUE REPORT")
    display(missing_value_report(df))

    duplicate_count = df.duplicated().sum()

    print("Exact duplicate records:", duplicate_count)

    quality_summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Duplicate Records": duplicate_count,
        "Missing Cells": df.isna().sum().sum()
    })


quality_summary_df = pd.DataFrame(quality_summary)

print("\nOVERALL DATA QUALITY SUMMARY")
display(quality_summary_df)


SALES — MISSING VALUE REPORT


,Missing Count,Missing Percentage,Data Type
Delivery Date,49719,79.06,object
Order Number,0,0.00,int64
Line Item,0,0.00,int64
Order Date,0,0.00,object
CustomerKey,0,0.00,int64
StoreKey,0,0.00,int64
ProductKey,0,0.00,int64
Quantity,0,0.00,int64
Currency Code,0,0.00,object


Exact duplicate records: 0

CUSTOMERS — MISSING VALUE REPORT


,Missing Count,Missing Percentage,Data Type
State Code,10,0.07,object
CustomerKey,0,0.00,int64
Gender,0,0.00,object
Name,0,0.00,object
City,0,0.00,object
State,0,0.00,object
Zip Code,0,0.00,object
Country,0,0.00,object
Continent,0,0.00,object
Birthday,0,0.00,object


Exact duplicate records: 0

PRODUCTS — MISSING VALUE REPORT


,Missing Count,Missing Percentage,Data Type
ProductKey,0,0.00,int64
Product Name,0,0.00,object
Brand,0,0.00,object
Color,0,0.00,object
Unit Cost USD,0,0.00,object
Unit Price USD,0,0.00,object
SubcategoryKey,0,0.00,int64
Subcategory,0,0.00,object
CategoryKey,0,0.00,int64
Category,0,0.00,object


... (2 outputs left out)

Exact duplicate records: 0

EXCHANGE_RATES — MISSING VALUE REPORT


,Missing Count,Missing Percentage,Data Type
Date,0,0.00,object
Currency,0,0.00,object
Exchange,0,0.00,float64


Exact duplicate records: 0

OVERALL DATA QUALITY SUMMARY


,Dataset,Rows,Columns,Duplicate Records,Missing Cells
0,sales,62884,9,0,49719
1,customers,15266,10,0,10
2,products,2517,10,0,0
3,stores,67,5,0,1
4,exchange_rates,11215,3,0,0


### 6.1 Potential Identifier Assessment

Identifier columns are examined to understand their uniqueness and completeness.

This is particularly important when preparing datasets for relational data modelling.

The assessment identifies columns containing terms such as ID, Key, Code or Number. These are potential identifiers and will be reviewed before establishing relationships between tables.

In [9]:
for name, df in raw_data.items():

    print(f"\n{name.upper()} — POTENTIAL IDENTIFIERS")

    identifier_columns = [
        col for col in df.columns
        if any(
            keyword in str(col).lower()
            for keyword in ["id", "key", "code", "number"]
        )
    ]

    if not identifier_columns:
        print("No potential identifier columns detected.")

    for col in identifier_columns:

        print(f"\nColumn: {col}")
        print("Unique values:", df[col].nunique(dropna=True))
        print("Missing values:", df[col].isna().sum())
        print("Duplicate values:", df[col].duplicated().sum())


SALES — POTENTIAL IDENTIFIERS

Column: Order Number
Unique values: 26326
Missing values: 0
Duplicate values: 36558

Column: CustomerKey
Unique values: 11887
Missing values: 0
Duplicate values: 50997

Column: StoreKey
Unique values: 58
Missing values: 0
Duplicate values: 62826

Column: ProductKey
Unique values: 2492
Missing values: 0
Duplicate values: 60392

Column: Currency Code
Unique values: 5
Missing values: 0
Duplicate values: 62879

CUSTOMERS — POTENTIAL IDENTIFIERS

Column: CustomerKey
Unique values: 15266
Missing values: 0
Duplicate values: 0

Column: State Code
Unique values: 467
Missing values: 10
Duplicate values: 14798

Column: Zip Code
Unique values: 9505
Missing values: 0
Duplicate values: 5761

PRODUCTS — POTENTIAL IDENTIFIERS

Column: ProductKey
Unique values: 2517
Missing values: 0
Duplicate values: 0

Column: SubcategoryKey
Unique values: 32
Missing values: 0
Duplicate values: 2485

Column: CategoryKey
Unique values: 8
Missing values: 0
Duplicate values: 2509

STORES 

## 7. Column Name Standardisation

Consistent column naming improves readability and reduces errors during data manipulation.

Column names are standardised using snake_case formatting.

The transformation includes:

- Removing leading and trailing spaces.
- Converting spaces and special characters into underscores.
- Separating words in camelCase column names.
- Converting column names to lowercase.
- Removing unnecessary unnamed columns.

The original datasets remain unchanged. All transformations are performed on working copies.

In [10]:
# Function to standardise column names
def standardise_column_name(column):

    column = str(column).strip()

    # Separate camelCase words
    column = re.sub(
        r"([a-z0-9])([A-Z])",
        r"\1_\2",
        column
    )

    # Replace special characters with underscores
    column = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        column
    )

    # Remove repeated underscores
    column = re.sub(r"_+", "_", column)

    return column.strip("_").lower()


# Create working copies
working_data = {}

for name, df in raw_data.items():

    cleaned = df.copy()

    cleaned.columns = [
        standardise_column_name(col)
        for col in cleaned.columns
    ]

    # Remove unnamed columns
    unnamed_columns = [
        col for col in cleaned.columns
        if not col or col.startswith("unnamed")
    ]

    cleaned = cleaned.drop(
        columns=unnamed_columns
    )

    working_data[name] = cleaned

    print(f"\n{name.upper()} — STANDARDISED COLUMNS")
    print(cleaned.columns.tolist())


SALES — STANDARDISED COLUMNS
['order_number', 'line_item', 'order_date', 'delivery_date', 'customer_key', 'store_key', 'product_key', 'quantity', 'currency_code']

CUSTOMERS — STANDARDISED COLUMNS
['customer_key', 'gender', 'name', 'city', 'state_code', 'state', 'zip_code', 'country', 'continent', 'birthday']

PRODUCTS — STANDARDISED COLUMNS
['product_key', 'product_name', 'brand', 'color', 'unit_cost_usd', 'unit_price_usd', 'subcategory_key', 'subcategory', 'category_key', 'category']

STORES — STANDARDISED COLUMNS
['store_key', 'country', 'state', 'square_meters', 'open_date']

EXCHANGE_RATES — STANDARDISED COLUMNS
['date', 'currency', 'exchange']


## 8. Text Cleaning and Missing-Value Standardisation

Text fields are inspected and standardised to improve consistency.

The following transformations are applied:

- Remove unnecessary leading and trailing whitespace.
- Convert common missing-value placeholders into proper missing values.
- Preserve meaningful text values and their original capitalisation.

Common placeholders include blank strings, N/A, NULL, NONE and dashes.

Missing values are not automatically replaced with zero because a missing observation does not necessarily represent zero sales, zero quantity or zero cost.

In [11]:
# Common missing-value markers
MISSING_MARKERS = {
    "",
    "na",
    "n/a",
    "null",
    "none",
    "nan",
    "-",
    "--"
}


def clean_text_values(df):

    result = df.copy()

    text_columns = result.select_dtypes(
        include=["object", "string"]
    ).columns

    for col in text_columns:

        values = result[col].astype("string").str.strip()

        missing_mask = values.str.lower().isin(
            MISSING_MARKERS
        )

        result[col] = values.mask(
            missing_mask,
            pd.NA
        )

    return result


# Apply text cleaning
for name in working_data:

    working_data[name] = clean_text_values(
        working_data[name]
    )

print("Text standardisation completed.")

Text standardisation completed.


## 9. Date Format Standardisation

Date fields are identified and converted into Pandas datetime format.

This is important for subsequent time-series analysis, monthly sales comparisons and trend visualisation.

Date conversion is restricted to columns whose names suggest that they contain dates.

Unparseable values are converted to missing values and reported for investigation rather than silently ignored.

In [13]:
def convert_date_columns(df):

    result = df.copy()

    date_columns = [
        col for col in result.columns
        if any(
            keyword in col.lower()
            for keyword in ["date", "dob", "birth"]
        )
    ]

    reports = []

    for col in date_columns:

        original_non_null = result[col].notna().sum()

        converted = pd.to_datetime(
            result[col],
            errors="coerce"
        )

        valid_dates = converted.notna().sum()

        reports.append({
            "Column": col,
            "Original Non-Null Values": original_non_null,
            "Valid Dates": valid_dates,
            "Unparsed Values": original_non_null - valid_dates
        })

        result[col] = converted

    return result, pd.DataFrame(reports)


date_conversion_reports = {}

for name in working_data:

    working_data[name], report = convert_date_columns(
        working_data[name]
    )

    date_conversion_reports[name] = report

    if not report.empty:

        print(f"\n{name.upper()} — DATE CONVERSION REPORT")
        display(report)


SALES — DATE CONVERSION REPORT


,Column,Original Non-Null Values,Valid Dates,Unparsed Values
0,order_date,62884,62884,0
1,delivery_date,13165,13165,0



CUSTOMERS — DATE CONVERSION REPORT


,Column,Original Non-Null Values,Valid Dates,Unparsed Values
0,birthday,15266,15266,0



STORES — DATE CONVERSION REPORT


,Column,Original Non-Null Values,Valid Dates,Unparsed Values
0,open_date,67,67,0



EXCHANGE_RATES — DATE CONVERSION REPORT


,Column,Original Non-Null Values,Valid Dates,Unparsed Values
0,date,11215,11215,0


## 10. Duplicate Record Removal

Exact duplicate records are identified and removed from the working datasets.

A record is considered an exact duplicate when all its column values match another record.

Only exact duplicates are removed automatically.

Records sharing customer IDs, product IDs or other identifiers are retained because these may represent legitimate business relationships or repeated transactions.

The number of records removed is documented for transparency.

In [14]:
duplicate_reports = []

for name, df in working_data.items():

    duplicates_before = df.duplicated().sum()

    cleaned = df.drop_duplicates().copy()

    duplicates_after = cleaned.duplicated().sum()

    working_data[name] = cleaned

    duplicate_reports.append({

        "Dataset": name,

        "Duplicates Before": duplicates_before,

        "Records Removed": len(df) - len(cleaned),

        "Duplicates After": duplicates_after,

        "Remaining Records": len(cleaned)

    })


duplicate_report_df = pd.DataFrame(duplicate_reports)

display(duplicate_report_df)

,Dataset,Duplicates Before,Records Removed,Duplicates After,Remaining Records
0,sales,0,0,0,62884
1,customers,0,0,0,15266
2,products,0,0,0,2517
3,stores,0,0,0,67
4,exchange_rates,0,0,0,11215


## 11. Data Validation

After cleaning, the datasets undergo structural validation to establish whether the preparation process was successful.

The validation checks include:

- Confirming that datasets contain records.
- Checking for duplicate column names.
- Confirming that exact duplicate rows have been removed.
- Reviewing remaining missing values.
- Comparing the number of records before and after cleaning.

Missing values are reported rather than automatically eliminated. Their treatment will depend on their meaning and relevance to subsequent business analysis.

Successful validation establishes that the datasets are structurally prepared for further investigation.

In [15]:
validation_results = []

for name, df in working_data.items():

    validation_results.append({

        "Dataset": name,

        "Rows": len(df),

        "Columns": len(df.columns),

        "Unique Column Names": df.columns.is_unique,

        "Remaining Duplicates": df.duplicated().sum(),

        "Missing Cells": df.isna().sum().sum(),

        "Empty Dataset": df.empty

    })


validation_df = pd.DataFrame(validation_results)

display(validation_df)


# Validation assertions
assert validation_df["Unique Column Names"].all(), \
    "Duplicate column names detected."

assert (validation_df["Remaining Duplicates"] == 0).all(), \
    "Exact duplicate records remain."

assert (~validation_df["Empty Dataset"]).all(), \
    "One or more datasets are empty."

print("Structural validation completed successfully.")

,Dataset,Rows,Columns,Unique Column Names,Remaining Duplicates,Missing Cells,Empty Dataset
0,sales,62884,9,True,0,49719,False
1,customers,15266,10,True,0,10,False
2,products,2517,10,True,0,0,False
3,stores,67,5,True,0,1,False
4,exchange_rates,11215,3,True,0,0,False


Structural validation completed successfully.


## 12. Raw versus Processed Data Comparison

A comparison is performed between the original datasets and their cleaned versions.

This provides a transparent record of how the data preparation process affected the datasets.

The comparison includes:

- Original and processed row counts.
- Number of records removed.
- Original and processed column counts.

The results help establish the extent of data transformation and identify any unexpected changes.

In [16]:
comparison_results = []

for name in raw_data:

    original = raw_data[name]

    processed = working_data[name]

    comparison_results.append({

        "Dataset": name,

        "Original Rows": len(original),

        "Processed Rows": len(processed),

        "Rows Removed": len(original) - len(processed),

        "Original Columns": original.shape[1],

        "Processed Columns": processed.shape[1]

    })


comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)

,Dataset,Original Rows,Processed Rows,Rows Removed,Original Columns,Processed Columns
0,sales,62884,62884,0,9,9
1,customers,15266,15266,0,10,10
2,products,2517,2517,0,10,10
3,stores,67,67,0,5,5
4,exchange_rates,11215,11215,0,3,3


## 13. Exporting Processed Datasets

The cleaned datasets are exported into the project's `data/processed/` directory.

The original source files remain unchanged in `data/raw/`.

Each processed dataset is saved as a CSV file using a consistent naming convention.

These files will serve as the input for the subsequent exploratory data analysis notebook and the Power BI dashboard.

Exporting the processed datasets also makes the workflow reproducible and separates data preparation from subsequent analytical activities.

In [17]:
for name, df in working_data.items():

    output_file = PROCESSED_DIR / f"{name}_clean.csv"

    df.to_csv(
        output_file,
        index=False
    )

    print(f"Exported: {output_file.name}")

print("\nAll processed datasets exported successfully.")

Exported: sales_clean.csv
Exported: customers_clean.csv
Exported: products_clean.csv
Exported: stores_clean.csv
Exported: exchange_rates_clean.csv

All processed datasets exported successfully.


## 14. Final Data Preparation Summary

This notebook completed the initial data preparation and cleaning stage of the Retail Sales Intelligence project.

The process involved:

1. Loading and inspecting the five source datasets.
2. Assessing data quality and completeness.
3. Standardising column names and text values.
4. Converting identifiable date fields.
5. Identifying and removing exact duplicate records.
6. Validating the processed datasets.
7. Exporting cleaned datasets for subsequent analysis.

### Key Considerations

Although structural cleaning has been completed, remaining missing values and potential business-specific inconsistencies must be evaluated in context.

Further analytical checks will be conducted during exploratory data analysis.

### Next Stage

The next notebook, **02. Exploratory Data Analysis**, will investigate:

- Sales and revenue distributions.
- Monthly and yearly sales trends.
- Product and category performance.
- Customer purchasing behaviour.
- Geographical sales performance.
- Potential commercial opportunities.

The findings will inform the development of the Power BI dashboard and management recommendations.

In [18]:
# Final summary report

final_summary = []

for name, df in working_data.items():

    final_summary.append({

        "Dataset": name,

        "Total Records": len(df),

        "Total Columns": len(df.columns),

        "Missing Values": df.isna().sum().sum(),

        "Duplicate Records": df.duplicated().sum(),

        "Memory Usage (MB)": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2),
            2
        )

    })


final_summary_df = pd.DataFrame(final_summary)

display(final_summary_df)


print("=" * 60)
print("NOTEBOOK 01 — DATA PREPARATION COMPLETED")
print("=" * 60)

print(f"Processed datasets saved in: {PROCESSED_DIR}")

print("\nDatasets available for further analysis:")

for file in PROCESSED_DIR.glob("*.csv"):
    print("-", file.name)

,Dataset,Total Records,Total Columns,Missing Values,Duplicate Records,Memory Usage (MB)
0,sales,62884,9,49719,0,6.96
1,customers,15266,10,10,0,6.96
2,products,2517,10,0,0,1.12
3,stores,67,5,1,0,0.01
4,exchange_rates,11215,3,0,0,0.73


NOTEBOOK 01 — DATA PREPARATION COMPLETED
Processed datasets saved in: C:\Users\Administrator\retail-sales-intelligence\data\processed

Datasets available for further analysis:
- customers_clean.csv
- exchange_rates_clean.csv
- products_clean.csv
- sales_clean.csv
- stores_clean.csv
